# Exploración RGB de cutouts MaNGA — validación rápida

Notebook de trabajo para inspeccionar una muestra fija de 16 galaxias usando cutouts RGB de Legacy Survey. La descarga queda fijada en **DR11 (`ls-dr11`)**, **bandas `grz`**, **FOV factor 6** y **256 px**. Los archivos se guardan de forma aislada en `data/legacy_cutouts_playground/rgb_validation/`; no modifica la configuración de producción ni el dataset oficial.

El objetivo es iterar rápido: se comprueban rutas, esquema, IDs, jobs y legibilidad básica. Los errores individuales de descarga quedan registrados y no interrumpen el lote.

## 1. Dependencias y setup rápido del entorno

Las dependencias usadas aquí ya están declaradas en `requirements.txt`. La instalación queda **optativa**: solo activa `INSTALL_MISSING_PACKAGES` si este kernel no las tiene.

In [ ]:
import importlib.util
import subprocess
import sys

PACKAGES = {
    "requests": "requests",
    "PIL": "Pillow",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
    "numpy": "numpy",
    "scipy": "scipy",
}
INSTALL_MISSING_PACKAGES = False  # cambiar a True solo si falta alguna dependencia
missing_modules = [module for module in PACKAGES if importlib.util.find_spec(module) is None]
if missing_modules and INSTALL_MISSING_PACKAGES:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        *(PACKAGES[module] for module in missing_modules),
    ])
    missing_modules = [module for module in PACKAGES if importlib.util.find_spec(module) is None]
if missing_modules:
    missing_packages = ", ".join(PACKAGES[module] for module in missing_modules)
    raise ImportError(
        f"Faltan dependencias: {missing_packages}. Instálalas desde requirements.txt "
        "o activa INSTALL_MISSING_PACKAGES y vuelve a ejecutar esta celda."
    )

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage

print("Dependencias listas: requests, Pillow, matplotlib, numpy, pandas y scipy.")

## 2. Configuración base y checks de rutas

Se busca la raíz del repositorio a partir del directorio actual. Las rutas se comprueban al principio para que cualquier ajuste de ubicación falle con un mensaje legible.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "config.py").is_file() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "config.py").is_file():
    raise FileNotFoundError("No encontré la raíz del repositorio (se esperaba src/config.py).")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import MORFOLOGIA_CSV

NOTEBOOK_DIR = PROJECT_ROOT / "notebook" / "julio"
PLAYGROUND_DIR = PROJECT_ROOT / "data" / "legacy_cutouts_playground"
RGB_OUTPUT_DIR = PLAYGROUND_DIR / "rgb_validation" / "dr11_grz_fov6"
PLAYGROUND_INDEX_CSV = PLAYGROUND_DIR / "rgb_validation" / "playground_index.csv"
SUMMARY_CSV = PLAYGROUND_DIR / "rgb_validation" / "summary_by_experiment.csv"
QUALITY_METRICS_CSV = PLAYGROUND_DIR / "rgb_validation" / "quality_metrics.csv"

RANDOM_SEED = 42
SIZE_PX = 256
FOV_FACTOR = 6
LAYER = "ls-dr11"
BANDS = "grz"
PIXSCALE_MIN = 0.1
PIXSCALE_MAX = 0.6
MAX_CONCURRENT_REQUESTS = 2
MIN_REQUEST_INTERVAL_S = 1.0
MAX_RETRIES = 2
TIMEOUT_S = 60
OVERWRITE = False

# Parámetros exploratorios: se ajustan tras inspeccionar mapas, perfiles y casos.
CONFIG = {
    "bright_quantile": 0.99,
    "bright_quantile_line": 0.97,
    "chroma_threshold": 0.5,
    "saturation_threshold": 0.98,
    "line_span_threshold": 0.3,
    "line_aspect_threshold": 5.0,
    "line_z_threshold": 8.0,
    "outer_radius": 0.3,
    "outer_component_threshold": 0.01,
}

assert MORFOLOGIA_CSV.is_file(), f"No existe el CSV de morfología: {MORFOLOGIA_CSV}"
assert (NOTEBOOK_DIR / "playground_utils.py").is_file(), "No encuentro notebook/julio/playground_utils.py"
RGB_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"MORFOLOGIA_CSV : {MORFOLOGIA_CSV}")
print(f"RGB_OUTPUT_DIR : {RGB_OUTPUT_DIR}")
print(f"Configuración de descarga: {LAYER}, bands={BANDS}, FOV={FOV_FACTOR}, size={SIZE_PX}px")
print("Configuración de calidad:", CONFIG)

## 3. Carga de `morfologia.csv` y esquema mínimo

In [ ]:
from src.data.morfologia import derive_ttype_label, load_morfologia

REQUIRED_COLUMNS = {"name", "objra", "objdec", "PETRO_TH90", "Ttype"}
raw = load_morfologia(MORFOLOGIA_CSV)
missing_columns = sorted(REQUIRED_COLUMNS - set(raw.columns))
if missing_columns:
    raise ValueError(f"Faltan columnas obligatorias en morfologia.csv: {missing_columns}")

labeled = derive_ttype_label(raw)
print(f"Filas cargadas: {len(labeled):,}")
print("Columnas mínimas presentes:", sorted(REQUIRED_COLUMNS))
labeled[["name", "objra", "objdec", "PETRO_TH90", "Ttype", "label"]].head()

## 4. Filtros mínimos de calidad: nulos y rangos

El filtro se limita a las columnas necesarias para seleccionar y encuadrar cada cutout. Se reportan los conteos para poder ver rápidamente qué registros quedaron fuera.

In [ ]:
critical_columns = ["name", "objra", "objdec", "PETRO_TH90", "Ttype"]
quality = labeled.copy()
for column in ["objra", "objdec", "PETRO_TH90", "Ttype"]:
    quality[column] = pd.to_numeric(quality[column], errors="coerce")

n_before = len(quality)
quality = quality.dropna(subset=critical_columns).copy()
valid_ranges = (
    quality["objra"].between(0, 360)
    & quality["objdec"].between(-90, 90)
    & quality["PETRO_TH90"].gt(0)
    & quality["label"].isin(["temprana", "tardia"])
)
pool = quality.loc[valid_ranges].copy().reset_index(drop=True)

print(f"Filas iniciales                         : {n_before:,}")
print(f"Filas tras quitar nulos/no numéricos    : {len(quality):,}")
print(f"Filas tras rangos y etiquetas utilizables: {len(pool):,}")
print("Clases disponibles en pool:")
display(pool["label"].value_counts(dropna=False).rename_axis("label").to_frame("n"))

## 5. Muestra experimental y `sample_selected`

La lista conserva los 10 IDs del playground y agrega los 6 IDs nuevos. Se mantiene el orden original; los IDs no disponibles en el pool se reportan y se omiten de la muestra efectiva.

In [ ]:
sample_selected_ids = [
    "manga-8147-3704", "manga-8311-6103", "manga-8620-12704", "manga-8937-9101",
    "manga-8549-12701", "manga-8486-12701", "manga-8547-12703", "manga-8561-3704",
    "manga-7962-6102", "manga-7992-9101", "manga-10494-12703", "manga-10495-6101",
    "manga-10495-12704", "manga-10499-3701", "manga-10499-6104", "manga-10507-6103",
]
assert all(isinstance(name, str) and name.strip() for name in sample_selected_ids), "Cada ID debe ser texto no vacío."
duplicate_ids = sorted({name for name in sample_selected_ids if sample_selected_ids.count(name) > 1})
assert not duplicate_ids, f"IDs duplicados en sample_selected_ids: {duplicate_ids}"

pool_by_name = pool.drop_duplicates(subset=["name"], keep="first").set_index("name")
missing_ids = [name for name in sample_selected_ids if name not in pool_by_name.index]
valid_ids = [name for name in sample_selected_ids if name in pool_by_name.index]
sample_selected = pool_by_name.reindex(valid_ids).reset_index()
sample = sample_selected.copy()  # muestra fija: selección explícita y reproducible

print(f"IDs solicitados : {len(sample_selected_ids)}")
print(f"IDs encontrados : {len(sample_selected)}")
print(f"IDs faltantes   : {missing_ids if missing_ids else 'ninguno'}")
display(sample_selected[["name", "label", "Ttype", "PETRO_TH90", "objra", "objdec"]])

## 6. Smoke tests de utilidades

Casos rápidos para detectar errores obvios en el cálculo de `pixscale` y en la URL, sin probar conectividad ni características de todo el servicio.

In [ ]:
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))
from playground_utils import (
    build_cutout_url,
    compute_pixscale,
    download_batch,
    prepare_jobs,
)
from urllib.parse import parse_qs, urlparse

pixscale_cases = [
    compute_pixscale(1.0, fov_factor=FOV_FACTOR, size_px=SIZE_PX,
                     pixscale_min=PIXSCALE_MIN, pixscale_max=PIXSCALE_MAX),
    compute_pixscale(10.0, fov_factor=FOV_FACTOR, size_px=SIZE_PX,
                     pixscale_min=PIXSCALE_MIN, pixscale_max=PIXSCALE_MAX),
    compute_pixscale(100.0, fov_factor=FOV_FACTOR, size_px=SIZE_PX,
                     pixscale_min=PIXSCALE_MIN, pixscale_max=PIXSCALE_MAX),
]
assert all(PIXSCALE_MIN <= value <= PIXSCALE_MAX for value in pixscale_cases)
assert np.isclose(pixscale_cases[0], PIXSCALE_MIN)
assert np.isclose(pixscale_cases[-1], PIXSCALE_MAX)

url_test = build_cutout_url(
    ra=150.0, dec=2.5, pixscale=pixscale_cases[1],
    size_px=SIZE_PX, layer=LAYER, bands=BANDS,
)
query = parse_qs(urlparse(url_test).query)
for key in ["ra", "dec", "size", "layer", "pixscale", "bands"]:
    assert key in query and query[key], f"Falta el parámetro {key!r} en la URL."
assert query["layer"] == [LAYER] and query["bands"] == [BANDS]
url_without_bands = build_cutout_url(150.0, 2.5, pixscale_cases[1], size_px=SIZE_PX, layer=LAYER, bands=None)
assert "bands" not in parse_qs(urlparse(url_without_bands).query)

print("Smoke tests OK.")
print("pixscale para PETRO_TH90=1, 10, 100 arcsec:", [round(value, 4) for value in pixscale_cases])
print("URL de ejemplo:", url_test)

## 7. Validación previa de jobs

Cada galaxia válida produce un job con una URL y una ruta local dentro del directorio de esta exploración.

In [ ]:
jobs = prepare_jobs(
    sample_df=sample_selected,
    playground_dir=PLAYGROUND_DIR,
    experiment="rgb_validation",
    subfolder="dr11_grz_fov6",
    layer=LAYER,
    bands=BANDS,
    fov_factor=FOV_FACTOR,
    size_px=SIZE_PX,
    pixscale_min=PIXSCALE_MIN,
    pixscale_max=PIXSCALE_MAX,
)

JOB_REQUIRED_FIELDS = {"name", "url", "local_path", "fov_factor", "pixscale"}
invalid_jobs = []
for job in jobs:
    missing_fields = JOB_REQUIRED_FIELDS - set(job)
    if missing_fields or not str(job.get("url", "")).strip():
        invalid_jobs.append((job.get("name"), sorted(missing_fields), "URL vacía" if not job.get("url") else None))
    if Path(job["local_path"]).parent.resolve() != RGB_OUTPUT_DIR.resolve():
        invalid_jobs.append((job.get("name"), [], "ruta local fuera del directorio esperado"))
    if not (PIXSCALE_MIN <= job["pixscale"] <= PIXSCALE_MAX):
        invalid_jobs.append((job.get("name"), [], "pixscale fuera de límites"))

if invalid_jobs:
    raise ValueError(f"Jobs inválidos (primeros casos): {invalid_jobs[:5]}")
print(f"Jobs listos: {len(jobs)}; todos tienen campos y rutas básicas válidas.")
if jobs:
    display(pd.DataFrame(jobs)[["name", "layer", "bands_or_color_mode", "fov_factor", "pixscale", "local_path"]])

## 8. Descarga con tolerancia básica a fallos

Se limita la concurrencia y se espacian las peticiones para tratar con moderación el servicio público. La utilidad compartida reintenta fallos de red/HTTP y registra cada resultado como `downloaded`, `cached` o `error`.

In [ ]:
if jobs:
    index_rgb = download_batch(
        jobs,
        max_concurrent=MAX_CONCURRENT_REQUESTS,
        min_request_interval_s=MIN_REQUEST_INTERVAL_S,
        overwrite=OVERWRITE,
        timeout_s=TIMEOUT_S,
        max_retries=MAX_RETRIES,
        backoff_s=2.0,
        backoff_429_s=10.0,
        verbose=True,
    )
else:
    index_rgb = pd.DataFrame()
    print("No hay jobs para descargar: revisa los IDs encontrados en la sección 5.")

if not index_rgb.empty:
    display(index_rgb[["name", "status", "http_status", "bytes", "error"]])

## 9. Validación post-descarga, integridad e inspección RGB

Se comprueba la existencia de las rutas y se abre una pequeña muestra con `PIL.Image.verify()`. Después se visualiza la composición RGB recibida y se resumen distribuciones de canales; estos valores describen el JPEG estirado del visor, no flujos fotométricos calibrados.

In [ ]:
if not index_rgb.empty:
    index_rgb["file_exists"] = index_rgb["local_path"].map(lambda path: Path(path).is_file())
    index_rgb["bytes_non_null"] = index_rgb["bytes"].notna()
    missing_files = index_rgb.loc[~index_rgb["file_exists"], "name"].tolist()
    print(f"Filas en índice                 : {len(index_rgb)}")
    print(f"Archivos presentes              : {int(index_rgb['file_exists'].sum())}")
    print(f"Rutas sin archivo               : {missing_files if missing_files else 'ninguna'}")
    print(f"Filas con bytes no nulos        : {int(index_rgb['bytes_non_null'].sum())}")
    print("Conteo por estado:")
    display(index_rgb["status"].value_counts(dropna=False).rename_axis("status").to_frame("n"))

    existing_rows = index_rgb[index_rgb["file_exists"]].copy()
    verify_failures = []
    for _, row in existing_rows.head(3).iterrows():
        try:
            with Image.open(row["local_path"]) as image:
                image.verify()
        except Exception as error:
            verify_failures.append((row["name"], str(error)))
    print(f"Imágenes verificadas con PIL (muestra de hasta 3): {min(3, len(existing_rows))}")
    print(f"Fallos de verify: {verify_failures if verify_failures else 'ninguno'}")

    rgb_arrays = []
    rgb_records = []
    image_failures = []
    for _, row in existing_rows.iterrows():
        try:
            with Image.open(row["local_path"]) as image:
                rgb = np.asarray(image.convert("RGB"), dtype=np.uint8)
        except Exception as error:
            image_failures.append((row["name"], str(error)))
            continue
        rgb_arrays.append((row["name"], row.get("label"), row.get("Ttype"), rgb))
        rgb_records.append({
            "name": row["name"],
            "label": row.get("label"),
            "Ttype": row.get("Ttype"),
            "width": rgb.shape[1],
            "height": rgb.shape[0],
            "mean_R": float(rgb[:, :, 0].mean()),
            "mean_G": float(rgb[:, :, 1].mean()),
            "mean_B": float(rgb[:, :, 2].mean()),
            "std_R": float(rgb[:, :, 0].std()),
            "std_G": float(rgb[:, :, 1].std()),
            "std_B": float(rgb[:, :, 2].std()),
        })
    rgb_stats = pd.DataFrame(rgb_records)
    print(f"Imágenes no legibles al inspeccionarlas: {image_failures if image_failures else 'ninguna'}")

    if rgb_arrays:
        ncols = 4
        nrows = int(np.ceil(len(rgb_arrays) / ncols))
        fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)
        for ax, (name, label, ttype, rgb) in zip(axes.flat, rgb_arrays):
            ax.imshow(rgb)
            ax.set_title(f"{name}\n{label} (T={ttype})", fontsize=8)
            ax.axis("off")
        for ax in axes.flat[len(rgb_arrays):]:
            ax.axis("off")
        fig.suptitle("Cutouts Legacy Survey — composición RGB / grz", fontsize=14)
        plt.tight_layout()
        plt.show()

        fig, axes = plt.subplots(1, 3, figsize=(14, 4))
        for channel_i, (channel_name, color) in enumerate(zip(["R", "G", "B"], ["red", "green", "blue"])):
            values = np.concatenate([rgb[:, :, channel_i].ravel() for _, _, _, rgb in rgb_arrays])
            axes[channel_i].hist(values, bins=32, color=color, alpha=0.8)
            axes[channel_i].set_title(f"Canal {channel_name}")
            axes[channel_i].set_xlabel("Intensidad JPEG (0–255)")
            axes[channel_i].set_ylabel("Píxeles")
        fig.suptitle("Distribución de intensidades por canal RGB — muestra disponible")
        plt.tight_layout()
        plt.show()
        display(rgb_stats.head(16))
else:
    rgb_stats = pd.DataFrame()
    print("No hay índice de descargas para validar. Ejecuta la sección 8 primero.")

## 10. Exportación y reporte de estado

Se guardan el índice auditable y un resumen compacto. El CSV incluye parámetros de descarga, estado, rutas y métricas RGB cuando la imagen pudo abrirse.

In [ ]:
if not index_rgb.empty:
    rgb_metrics = rgb_stats.drop(columns=["label", "Ttype"], errors="ignore")
    index_rgb = index_rgb.merge(rgb_metrics, on="name", how="left")
    status_counts = index_rgb.groupby(["experiment", "status"], dropna=False).size().unstack(fill_value=0)
    summary_by_experiment = index_rgb.groupby("experiment", dropna=False).agg(
        jobs=("name", "size"),
        bytes_total=("bytes", "sum"),
        files_present=("file_exists", "sum"),
    ).join(status_counts).reset_index()
else:
    summary_by_experiment = pd.DataFrame(
        columns=["experiment", "jobs", "bytes_total", "files_present", "downloaded", "cached", "error"]
    )

PLAYGROUND_INDEX_CSV.parent.mkdir(parents=True, exist_ok=True)
index_rgb.to_csv(PLAYGROUND_INDEX_CSV, index=False)
summary_by_experiment.to_csv(SUMMARY_CSV, index=False)

print(f"Índice guardado : {PLAYGROUND_INDEX_CSV}")
print(f"Resumen guardado: {SUMMARY_CSV}")
if not summary_by_experiment.empty:
    display(summary_by_experiment)
else:
    print("Resumen vacío: no hubo resultados de descarga que exportar.")

# Playground de calidad visual RGB

A partir de aquí el foco es distinto de la descarga: **inspeccionar -> medir -> visualizar -> comparar -> formular reglas**. El análisis estudia por separado calidad visual, contaminación brillante y saturación; una posible mala escala se conserva como una pregunta independiente. Las reglas serán heurísticas explicables, no probabilidades ni un clasificador entrenado.

## 11. Carga de imágenes y etiquetas manuales opcionales

La misma estructura funciona con pocas imágenes. Inicialmente todas se etiquetan como `unknown`; después de la inspección visual se pueden cambiar a `good`, `line_artifact`, `bright_contamination` o `saturation`. Estas etiquetas sirven únicamente para comparar métricas y revisar falsos positivos o negativos.

In [ ]:
def load_rgb_image(path):
    """Carga una imagen RGB como arreglo float32 normalizado a [0, 1]."""
    with Image.open(path) as image:
        return np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0

MANUAL_LABELS = {
    "manga-10494-12703": "bright_contamination",
    "manga-8147-3704":"good", 
    "manga-8311-6103":"good", 
    "manga-8620-12704":"good", 
    "manga-8937-9101":"bright_contamination",
    "manga-8549-12701":"bright_contamination", 
    "manga-8486-12701":"line_artifact", 
    "manga-8547-12703":"line_artifact", 
    "manga-8561-3704":"good",
    "manga-7962-6102":"line_artifact", 
    "manga-7992-9101":"line_artifact", 
    "manga-10494-12703":"bright_contamination", 
    "manga-10495-6101":"line_artifact",
    "manga-10495-12704":"bright_contamination", 
    "manga-10499-3701":"line_artifact", 
    "manga-10499-6104":"line_artifact", 
    "manga-10507-6103":"line_artifact"
}
VALID_MANUAL_LABELS = {"good", "line_artifact", "bright_contamination", "saturation", "unknown"}

image_rows = []
for path in sorted(RGB_OUTPUT_DIR.glob("*.jpg")):
    image_rows.append({
        "filename": path.name,
        "name": path.stem,
        "path": path,
        "manual_label": MANUAL_LABELS.get(path.stem, "unknown"),
    })
samples = pd.DataFrame(image_rows)
if not samples.empty:
    samples["manual_label"] = samples["manual_label"].where(
        samples["manual_label"].isin(VALID_MANUAL_LABELS), "unknown"
    )

print(f"Imágenes RGB disponibles: {len(samples)}")
if not samples.empty:
    example_img = load_rgb_image(samples.iloc[0]["path"])
    print("shape:", example_img.shape, " dtype:", example_img.dtype,
          " min:", example_img.min(), " max:", example_img.max())
    print("H x W x 3: altura, anchura y canales R/G/B.")
    display(samples)
else:
    print("No hay JPEGs aún. Ejecuta la descarga de la sección 8; el resto del playground queda listo.")

## 12. Representaciones derivadas y ejemplos sintéticos

Para detectar estructuras de un solo color usamos $I(x,y)=\max(R,G,B)$, porque una línea roja, verde o azul intensa sigue siendo brillante en este mapa. La cromaticidad $C=(\max-\min)/(\max+\epsilon)$ no significa por sí sola anomalía: aporta evidencia cuando coincide con intensidad alta y una geometría sospechosa.

In [ ]:
def intensity_map(img):
    return img.max(axis=2)


def chroma_map(img, eps=1e-6):
    maximum = img.max(axis=2)
    return (maximum - img.min(axis=2)) / (maximum + eps)


def make_synthetic_examples(size=128, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    base = np.clip(rng.normal(0.05, 0.015, (size, size, 3)), 0, 1).astype(np.float32)
    examples = {"ruido tenue": base.copy()}

    horizontal = base.copy()
    horizontal[size // 2 - 2:size // 2 + 2, 8:-8, 0] = 1.0
    examples["línea horizontal roja"] = horizontal

    vertical = base.copy()
    vertical[8:-8, size // 3 - 2:size // 3 + 2, 1] = 1.0
    examples["línea vertical verde"] = vertical

    bright_edge = base.copy()
    yy, xx = np.ogrid[:size, :size]
    bright_edge[(yy - size // 5) ** 2 + (xx - 4 * size // 5) ** 2 < (size // 10) ** 2] = 1.0
    examples["fuente brillante externa"] = bright_edge
    return examples


synthetic_examples = make_synthetic_examples()
fig, axes = plt.subplots(1, len(synthetic_examples), figsize=(16, 4))
for ax, (title, synthetic_img) in zip(axes, synthetic_examples.items()):
    ax.imshow(synthetic_img)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
fig.suptitle("Ejemplos sintéticos: sirven para entender métricas, no para simular galaxias")
fig.tight_layout()
plt.show()

In [ ]:
def plot_gallery(sample_df, ncols=4):
    """Muestra imágenes reales disponibles con etiqueta manual y dimensiones."""
    if sample_df.empty:
        print("No hay imágenes para mostrar.")
        return
    nrows = int(np.ceil(len(sample_df) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)
    for ax, (_, sample_row) in zip(axes.flat, sample_df.iterrows()):
        img = load_rgb_image(sample_row["path"])
        ax.imshow(img)
        ax.set_title(
            f"{sample_row['name']}\n{sample_row['manual_label']} | {img.shape[0]}x{img.shape[1]}",
            fontsize=8,
        )
        ax.axis("off")
    for ax in axes.flat[len(sample_df):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()


plot_gallery(samples)

In [ ]:
def show_rgb_representations(img, title="Imagen"):
    intensity = intensity_map(img)
    mean_intensity = img.mean(axis=2)
    chroma = chroma_map(img)
    fig, axes = plt.subplots(2, 4, figsize=(15, 7))
    views = [
        (img, "RGB", None), (img[:, :, 0], "R", "Reds"),
        (img[:, :, 1], "G", "Greens"), (img[:, :, 2], "B", "Blues"),
        (intensity, "max(R, G, B)", "gray"),
        (mean_intensity, "(R + G + B) / 3", "gray"),
        (chroma, "Cromaticidad", "magma"),
    ]
    for ax, (data, subtitle, cmap) in zip(axes.flat, views):
        ax.imshow(data, cmap=cmap, vmin=0 if cmap else None, vmax=1 if cmap else None)
        ax.set_title(subtitle)
        ax.axis("off")
    axes.flat[-1].axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


if not samples.empty:
    show_rgb_representations(load_rgb_image(samples.iloc[0]["path"]), samples.iloc[0]["name"])
else:
    show_rgb_representations(synthetic_examples["línea horizontal roja"], "Ejemplo sintético")

## 13. Máscaras brillantes y estadísticas robustas

La media puede quedar dominada por una estrella o una línea. Por eso usamos mediana y MAD: $MAD=\mathrm{mediana}(|x-\mathrm{mediana}(x)|)$, y un z-score robusto basado en $1.4826\,MAD$. El umbral brillante se explora por cuantiles; no se fija como una verdad física.

In [ ]:
def robust_zscore(values, eps=1e-8):
    """Z-score robusto centrado en la mediana; usa desviación estándar si MAD es cero."""
    values = np.asarray(values, dtype=float)
    median = np.median(values)
    mad = np.median(np.abs(values - median))
    scale = 1.4826 * mad
    if scale < eps:
        scale = values.std()
    if scale < eps:
        return np.zeros_like(values)
    return (values - median) / scale


def masks_from_image(img, config=CONFIG):
    intensity = intensity_map(img)
    chroma = chroma_map(img)
    threshold = np.quantile(intensity, config["bright_quantile"])
    bright_mask = intensity >= threshold
    colorful_mask = chroma >= config["chroma_threshold"]
    saturated_mask = intensity >= config["saturation_threshold"]
    return intensity, chroma, bright_mask, colorful_mask, bright_mask & colorful_mask, saturated_mask

def line_mask_from_image(img, config=CONFIG):
    intensity = intensity_map(img)
    threshold = np.quantile(intensity, config["bright_quantile_line"])
    return intensity >= threshold

def plot_masks_for_quantiles(img, quantiles=(0.95, 0.97, 0.98, 0.99, 0.995)):
    intensity = intensity_map(img)
    fig, axes = plt.subplots(1, len(quantiles) + 1, figsize=(3 * (len(quantiles) + 1), 3))
    axes[0].imshow(img)
    axes[0].set_title("RGB")
    axes[0].axis("off")
    for ax, quantile in zip(axes[1:], quantiles):
        ax.imshow(intensity >= np.quantile(intensity, quantile), cmap="gray")
        ax.set_title(f"q={quantile}")
        ax.axis("off")
    fig.tight_layout()
    plt.show()


reference_img = load_rgb_image(samples.iloc[0]["path"]) if not samples.empty else synthetic_examples["línea horizontal roja"]
plot_masks_for_quantiles(reference_img)

## 14. Perfiles por filas, columnas y continuidad

Una línea horizontal deja un pico en las fracciones brillantes por fila; una vertical deja un pico por columna. `longest_true_run` comprueba si el pico corresponde a una estructura continua: una estrella local suele producir un pico corto, mientras una línea puede abarcar gran parte del eje.

In [ ]:
def longest_true_run(mask_1d):
    """Longitud de la secuencia consecutiva más larga de valores True."""
    longest = current = 0
    for value in np.asarray(mask_1d, dtype=bool):
        current = current + 1 if value else 0
        longest = max(longest, current)
    return longest


def profile_metrics(img, config=CONFIG):
    intensity, chroma, _, colorful, _, _ = masks_from_image(img, config)
    bright = line_mask_from_image(img, config)
    bright_color = bright & (chroma >= config["chroma_threshold"])
    return {
        "row_mean_intensity": intensity.mean(axis=1),
        "col_mean_intensity": intensity.mean(axis=0),
        "row_max_intensity": intensity.max(axis=1),
        "col_max_intensity": intensity.max(axis=0),
        "row_bright_fraction": bright.mean(axis=1),
        "col_bright_fraction": bright.mean(axis=0),
        "row_color_fraction": colorful.mean(axis=1),
        "col_color_fraction": colorful.mean(axis=0),
        "row_bright_color_fraction": bright_color.mean(axis=1),
        "col_bright_color_fraction": bright_color.mean(axis=0),
        "row_longest_bright_run": np.array([longest_true_run(row) / bright.shape[1] for row in bright]),
        "col_longest_bright_run": np.array([longest_true_run(column) / bright.shape[0] for column in bright.T]),
    }


def plot_profiles(img, title="Perfiles"):
    profiles = profile_metrics(img)
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes[0, 0].imshow(img)
    axes[0, 0].set_title("RGB")
    axes[0, 0].axis("off")
    for values, label in [
        (profiles["row_bright_fraction"], "brillante"),
        (profiles["row_bright_color_fraction"], "brillante y color"),
        (profiles["row_longest_bright_run"], "run continuo"),
    ]:
        axes[0, 1].plot(values, label=label)
    axes[0, 1].set_title("Perfiles por filas")
    axes[0, 1].set_xlabel("fila")
    axes[0, 1].legend()
    for values, label in [
        (profiles["col_bright_fraction"], "brillante"),
        (profiles["col_bright_color_fraction"], "brillante y color"),
        (profiles["col_longest_bright_run"], "run continuo"),
    ]:
        axes[1, 0].plot(values, label=label)
    axes[1, 0].set_title("Perfiles por columnas")
    axes[1, 0].set_xlabel("columna")
    axes[1, 0].legend()
    axes[1, 1].plot(robust_zscore(profiles["row_bright_fraction"]), label="filas")
    axes[1, 1].plot(robust_zscore(profiles["col_bright_fraction"]), label="columnas")
    axes[1, 1].axhline(CONFIG["line_z_threshold"], color="crimson", linestyle="--", label="threshold exploratorio")
    axes[1, 1].set_title("Z-score robusto de fracciones brillantes")
    axes[1, 1].legend()
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


plot_profiles(synthetic_examples["línea horizontal roja"], "Cadena de evidencia: línea horizontal sintética")

## 15. Componentes conectados y contaminación externa

Las máscaras se separan en objetos conectados. De cada objeto medimos área, caja envolvente, elongación, extensión, brillo y distancia al centro. Una galaxia edge-on puede ser alargada, por eso una alarma lineal requiere combinar elongación con extensión, grosor, intensidad y posición.

In [ ]:
def distance_map(shape):
    height, width = shape[:2]
    yy, xx = np.indices((height, width))
    radius = np.sqrt((yy - (height - 1) / 2) ** 2 + (xx - (width - 1) / 2) ** 2)
    return radius / radius.max()


def component_table(mask, intensity):
    labels, n_components = ndimage.label(mask)
    height, width = mask.shape
    records = []
    for component_id in range(1, n_components + 1):
        ys, xs = np.where(labels == component_id)
        if len(ys) == 0:
            continue
        y0, y1 = ys.min(), ys.max() + 1
        x0, x1 = xs.min(), xs.max() + 1
        component_height, component_width = y1 - y0, x1 - x0
        short_axis = min(component_width, component_height)
        centroid_y, centroid_x = ys.mean(), xs.mean()
        center_distance = np.sqrt(
            ((centroid_y - (height - 1) / 2) / height) ** 2
            + ((centroid_x - (width - 1) / 2) / width) ** 2
        ) / np.sqrt(0.5**2 + 0.5**2)
        records.append({
            "component": component_id,
            "area": len(ys),
            "area_fraction": len(ys) / mask.size,
            "y0": y0, "y1": y1, "x0": x0, "x1": x1,
            "width": component_width,
            "height": component_height,
            "aspect_ratio": max(component_width, component_height) / max(short_axis, 1),
            "span": max(component_width / width, component_height / height),
            "thickness_fraction": short_axis / min(height, width),
            "centroid_y": centroid_y,
            "centroid_x": centroid_x,
            "distance_to_center": center_distance,
            "mean_intensity": float(intensity[labels == component_id].mean()),
            "max_intensity": float(intensity[labels == component_id].max()),
        })
    return pd.DataFrame(records), labels


def plot_components(img, title="Componentes brillantes"):
    intensity, _, bright_mask, _, bright_color_mask, _ = masks_from_image(img)
    components, _ = component_table(bright_mask, intensity)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img)
    axes[0].set_title("RGB")
    axes[1].imshow(bright_mask, cmap="gray")
    axes[1].set_title("Máscara brillante")
    axes[2].imshow(bright_color_mask, cmap="gray")
    axes[2].set_title("Brillante + color")
    for _, component in components.iterrows():
        rectangle = plt.Rectangle((component.x0, component.y0), component.width, component.height,
                                  fill=False, edgecolor="lime", linewidth=1.5)
        axes[0].add_patch(rectangle)
        axes[0].text(component.x0, component.y0, str(int(component.component)), color="lime", fontsize=8)
    for ax in axes:
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()
    return components


example_components = plot_components(synthetic_examples["línea horizontal roja"])
display(example_components.head())

## 16. Métricas resumidas por imagen

Aquí reunimos las mediciones sin decidir todavía una bandera final. Así podemos comparar distribuciones, buscar métricas redundantes y verificar si las imágenes etiquetadas manualmente se separan antes de elegir thresholds.

In [ ]:
def extract_quality_metrics(img, config=CONFIG):
    intensity, _, bright, _, _, saturated = masks_from_image(img, config)
    profiles = profile_metrics(img, config)
    bright_components, _ = component_table(bright, intensity)
    line_components, _ = component_table(line_mask_from_image(img, config), intensity)
    saturated_components, _ = component_table(saturated, intensity)
    radius = distance_map(img.shape)
    outer = radius >= config["outer_radius"]

    metrics = {
        "max_row_bright_fraction": float(profiles["row_bright_fraction"].max()),
        "max_col_bright_fraction": float(profiles["col_bright_fraction"].max()),
        "max_row_color_fraction": float(profiles["row_color_fraction"].max()),
        "max_col_color_fraction": float(profiles["col_color_fraction"].max()),
        "max_row_bright_color_fraction": float(profiles["row_bright_color_fraction"].max()),
        "max_col_bright_color_fraction": float(profiles["col_bright_color_fraction"].max()),
        "max_row_bright_z": float(robust_zscore(profiles["row_bright_fraction"]).max()),
        "max_col_bright_z": float(robust_zscore(profiles["col_bright_fraction"]).max()),
        "max_horizontal_run_fraction": float(profiles["row_longest_bright_run"].max()),
        "max_vertical_run_fraction": float(profiles["col_longest_bright_run"].max()),
        "saturated_fraction": float(saturated.mean()),
        "bright_fraction_center": float(bright[radius < 0.25].mean()),
        "bright_fraction_outer": float(bright[outer].mean()),
        "n_bright_components": len(bright_components),
        "largest_bright_component_fraction": 0.0,
        "largest_component_aspect_ratio": 0.0,
        "largest_component_span": 0.0,
        "largest_outer_component_fraction": 0.0,
        "largest_saturated_component_fraction": 0.0,
    }
    if not bright_components.empty:
        metrics["largest_bright_component_fraction"] = float(
            bright_components["area_fraction"].max()
        )
        metrics["largest_outer_component_fraction"] = float(
            bright_components.loc[
                bright_components["distance_to_center"] >= config["outer_radius"],
                "area_fraction",
            ].max()
            if (bright_components["distance_to_center"] >= config["outer_radius"]).any()
            else 0.0
        )

    if not line_components.empty:
        metrics["largest_component_aspect_ratio"] = float(
            line_components["aspect_ratio"].max()
        )
        metrics["largest_component_span"] = float(
            line_components["span"].max()
        )
    if not saturated_components.empty:
        metrics["largest_saturated_component_fraction"] = float(saturated_components["area_fraction"].max())
    return metrics


quality_records = []
for _, sample_row in samples.iterrows():
    image_metrics = extract_quality_metrics(load_rgb_image(sample_row["path"]))
    quality_records.append({**sample_row.drop("path").to_dict(), **image_metrics})
quality_df = pd.DataFrame(quality_records)

print(f"Imágenes analizadas: {len(quality_df)}")
display(quality_df.head())

## 17. Scores heurísticos, thresholds y razones

Los scores no son probabilidades calibradas. Solo condensan evidencia en tres preguntas independientes: ¿hay una línea?, ¿hay contaminación brillante externa? y ¿hay una región muy saturada en el RGB renderizado? Las razones conservan las métricas que motivan cada decisión.

In [ ]:
def clipped_ratio(value, threshold):
    return float(np.clip(value / max(threshold, 1e-8), 0, 1))


def analyze_image_quality(img, config=CONFIG):
    """Devuelve métricas, scores heurísticos y razones interpretables para una imagen RGB."""
    metrics = extract_quality_metrics(img, config)
    horizontal_evidence = max(
        clipped_ratio(metrics["max_horizontal_run_fraction"], config["line_span_threshold"]),
        clipped_ratio(metrics["max_row_bright_z"], config["line_z_threshold"]),
    )
    vertical_evidence = max(
        clipped_ratio(metrics["max_vertical_run_fraction"], config["line_span_threshold"]),
        clipped_ratio(metrics["max_col_bright_z"], config["line_z_threshold"]),
    )
    line_geometry = (
        metrics["largest_component_aspect_ratio"] >= config["line_aspect_threshold"]
        and metrics["largest_component_span"] >= config["line_span_threshold"]
    )
    line_score = max(horizontal_evidence, vertical_evidence) * (1.0 if line_geometry else 0.5)
    bright_score = max(
        clipped_ratio(metrics["largest_outer_component_fraction"], config["outer_component_threshold"]),
        clipped_ratio(metrics["bright_fraction_outer"], config["outer_component_threshold"]),
    )
    saturation_score = max(
        clipped_ratio(metrics["saturated_fraction"], 0.01),
        clipped_ratio(metrics["largest_saturated_component_fraction"], 0.005),
    )

    reasons = []
    if metrics["max_horizontal_run_fraction"] >= config["line_span_threshold"]:
        reasons.append(f"Estructura horizontal brillante continua: {metrics['max_horizontal_run_fraction']:.0%} del ancho.")
    if metrics["max_vertical_run_fraction"] >= config["line_span_threshold"]:
        reasons.append(f"Estructura vertical brillante continua: {metrics['max_vertical_run_fraction']:.0%} de la altura.")
    if line_geometry:
        reasons.append(
            f"Componente alargado: aspect={metrics['largest_component_aspect_ratio']:.1f}, "
            f"span={metrics['largest_component_span']:.0%}."
        )
    if metrics["largest_outer_component_fraction"] >= config["outer_component_threshold"]:
        reasons.append(f"Componente brillante externo grande: {metrics['largest_outer_component_fraction']:.1%} del frame.")
    if metrics["saturated_fraction"] >= 0.01:
        reasons.append(f"Fracción saturada RGB elevada: {metrics['saturated_fraction']:.1%} del frame.")
    if metrics["largest_saturated_component_fraction"] >= 0.005:
        reasons.append(f"Región saturada RGB grande: {metrics['largest_saturated_component_fraction']:.1%} del frame.")
    if not reasons:
        reasons.append("No hay evidencia fuerte con los thresholds exploratorios actuales.")

    line_artifact = bool(line_geometry and max(horizontal_evidence, vertical_evidence) >= 1.0)
    bright_contamination = bool(bright_score >= 1.0)
    saturation = bool(saturation_score >= 1.0)
    return {
        "is_suspicious": line_artifact or bright_contamination,
        "line_artifact": line_artifact,
        "line_score": line_score,
        "bright_contamination": bright_contamination,
        "bright_score": bright_score,
        "saturation": saturation,
        "saturation_score": saturation_score,
        "metrics": metrics,
        "reasons": reasons,
    }


if not samples.empty:
    demo_report = analyze_image_quality(load_rgb_image(samples.iloc[0]["path"]))
else:
    demo_report = analyze_image_quality(synthetic_examples["línea horizontal roja"])
print("Flags:", {key: demo_report[key] for key in ["is_suspicious", "line_artifact", "bright_contamination", "saturation"]})
print("Razones:")
for reason in demo_report["reasons"]:
    print("-", reason)

## 18. Ficha visual de diagnóstico

La ficha reúne la imagen, las representaciones, máscaras, componentes y perfiles. La intención es que cada bandera se pueda discutir mirando la evidencia, no confiar ciegamente en un número.

In [ ]:
def plot_quality_diagnostics(img, title="Diagnóstico", config=CONFIG):
    intensity, chroma, bright, colorful, bright_color, _ = masks_from_image(img, config)
    profiles = profile_metrics(img, config)
    components, _ = component_table(bright, intensity)
    report = analyze_image_quality(img, config)
    fig = plt.figure(figsize=(16, 14), constrained_layout=True)
    grid = fig.add_gridspec(4, 3, height_ratios=[1, 1, 0.8, 0.8])
    axes = [fig.add_subplot(grid[row, col]) for row in range(2) for col in range(3)]
    views = [
        (img, "RGB", None), (intensity, "Intensidad max RGB", "gray"), (chroma, "Cromaticidad", "magma"),
        (bright, "Máscara brillante", "gray"), (colorful, "Máscara de color", "gray"),
        (bright_color, "Brillante + color", "gray"),
    ]
    for ax, (data, subtitle, cmap) in zip(axes, views):
        ax.imshow(data, cmap=cmap, vmin=0 if cmap else None, vmax=1 if cmap else None)
        ax.set_title(subtitle)
        ax.axis("off")
    for _, component in components.iterrows():
        axes[0].add_patch(plt.Rectangle((component.x0, component.y0), component.width, component.height,
                                         fill=False, edgecolor="lime", linewidth=1.2))
    ax_rows = fig.add_subplot(grid[2, :])
    ax_rows.plot(profiles["row_bright_fraction"], label="bright fraction")
    ax_rows.plot(profiles["row_longest_bright_run"], label="longest run")
    ax_rows.plot(robust_zscore(profiles["row_bright_fraction"]), label="robust z")
    ax_rows.set_title("Perfiles por filas")
    ax_rows.legend(ncol=3)
    ax_cols = fig.add_subplot(grid[3, :])
    ax_cols.plot(profiles["col_bright_fraction"], label="bright fraction")
    ax_cols.plot(profiles["col_longest_bright_run"], label="longest run")
    ax_cols.plot(robust_zscore(profiles["col_bright_fraction"]), label="robust z")
    ax_cols.set_title("Perfiles por columnas")
    ax_cols.legend(ncol=3)
    fig.suptitle(f"{title}\n" + " | ".join(report["reasons"]), fontsize=11)
    plt.show()
    return report


def inspect_image_quality(path):
    img = load_rgb_image(path)
    report = plot_quality_diagnostics(img, title=Path(path).name)
    print("Scores:", {key: round(report[key], 3) for key in ["line_score", "bright_score", "saturation_score"]})
    print("Razones:")
    for reason in report["reasons"]:
        print("-", reason)
    return report


if not samples.empty:
    report = inspect_image_quality(samples.iloc[0]["path"])
else:
    report = plot_quality_diagnostics(synthetic_examples["línea horizontal roja"], "Ejemplo sintético")

## 19. Comparación, correlaciones y revisión de errores

Tras etiquetar manualmente algunos casos, esta sección permite comprobar qué métricas separan realmente las clases. También agrupa casos de verdaderos/falsos positivos y negativos para inspeccionar los errores del detector, en lugar de optimizar thresholds a ciegas.

In [ ]:
def evaluate_quality_table(sample_df, config=CONFIG):
    records = []
    for _, sample_row in sample_df.iterrows():
        report = analyze_image_quality(load_rgb_image(sample_row["path"]), config)
        records.append({
            **sample_row.drop("path").to_dict(),
            **report["metrics"],
            "line_flag": report["line_artifact"],
            "bright_flag": report["bright_contamination"],
            "saturation_flag": report["saturation"],
            "line_score": report["line_score"],
            "bright_score": report["bright_score"],
            "saturation_score": report["saturation_score"],
            "reasons": " | ".join(report["reasons"]),
        })
    return pd.DataFrame(records)


quality_df = evaluate_quality_table(samples) if not samples.empty else pd.DataFrame()
if not quality_df.empty:
    display(quality_df[["filename", "manual_label", "line_flag", "bright_flag", "saturation_flag", "line_score", "bright_score", "saturation_score"]])

    numeric = quality_df.select_dtypes(include="number")
    if numeric.shape[1] > 1:
        correlation = numeric.corr()
        fig, ax = plt.subplots(figsize=(12, 10))
        image = ax.imshow(correlation, vmin=-1, vmax=1, cmap="coolwarm")
        ax.set_xticks(range(len(correlation.columns)), correlation.columns, rotation=90, fontsize=7)
        ax.set_yticks(range(len(correlation.columns)), correlation.columns, fontsize=7)
        fig.colorbar(image, ax=ax, label="correlación")
        ax.set_title("Correlación entre métricas: buscar redundancias")
        plt.tight_layout()
        plt.show()

    labeled_cases = quality_df[quality_df["manual_label"] != "unknown"]
    if not labeled_cases.empty:
        fig, axes = plt.subplots(1, 3, figsize=(15, 4))
        for ax, metric in zip(axes, ["max_horizontal_run_fraction", "largest_outer_component_fraction", "saturated_fraction"]):
            labels = labeled_cases["manual_label"].unique()
            ax.boxplot([labeled_cases.loc[labeled_cases["manual_label"] == label, metric] for label in labels], tick_labels=labels)
            ax.set_title(metric)
            ax.tick_params(axis="x", rotation=30)
        plt.tight_layout()
        plt.show()
else:
    print("No hay imágenes reales para evaluar todavía.")

In [ ]:
def show_cases(case_df, title, ncols=4):
    if case_df.empty:
        print(f"{title}: no hay casos.")
        return
    paths = {row.filename: row.path for _, row in samples.iterrows()}
    nrows = int(np.ceil(len(case_df) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows), squeeze=False)
    for ax, (_, case) in zip(axes.flat, case_df.iterrows()):
        ax.imshow(load_rgb_image(paths[case["filename"]]))
        ax.set_title(f"{case['filename']}\nmanual={case['manual_label']}", fontsize=8)
        ax.axis("off")
    for ax in axes.flat[len(case_df):]:
        ax.axis("off")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


if not quality_df.empty and (quality_df["manual_label"] != "unknown").any():
    line_truth = quality_df["manual_label"].eq("line_artifact")
    case_groups = {
        "Líneas: verdaderos positivos": quality_df[line_truth & quality_df["line_flag"]],
        "Líneas: falsos positivos": quality_df[~line_truth & quality_df["line_flag"]],
        "Líneas: falsos negativos": quality_df[line_truth & ~quality_df["line_flag"]],
        "Líneas: verdaderos negativos": quality_df[~line_truth & ~quality_df["line_flag"]],
    }
    for title, cases in case_groups.items():
        show_cases(cases, title)
else:
    print("Etiqueta algunos registros en MANUAL_LABELS y vuelve a ejecutar desde la sección 11 para revisar errores.")

In [ ]:
# 19.1. Revisión de errores: contaminación brillante externa
# Excluye "unknown" para no considerarlo erróneamente como un negativo.
if not quality_df.empty:
    reviewed_cases = quality_df[quality_df["manual_label"] != "unknown"]

    if not reviewed_cases.empty:
        bright_truth = reviewed_cases["manual_label"].eq("bright_contamination")

        bright_case_groups = {
            "Contaminación brillante: verdaderos positivos": reviewed_cases[
                bright_truth & reviewed_cases["bright_flag"]
            ],
            "Contaminación brillante: falsos positivos": reviewed_cases[
                ~bright_truth & reviewed_cases["bright_flag"]
            ],
            "Contaminación brillante: falsos negativos": reviewed_cases[
                bright_truth & ~reviewed_cases["bright_flag"]
            ],
            "Contaminación brillante: verdaderos negativos": reviewed_cases[
                ~bright_truth & ~reviewed_cases["bright_flag"]
            ],
        }

        for title, cases in bright_case_groups.items():
            show_cases(cases, title)
    else:
        print("No hay etiquetas manuales revisadas para evaluar contaminación brillante.")
else:
    print("No hay imágenes reales para evaluar todavía.")

In [ ]:
# 19.2. Evaluación global de is_suspicious
if not quality_df.empty:
    reviewed_cases = quality_df[quality_df["manual_label"] != "unknown"].copy()

    if not reviewed_cases.empty:
        # Verdad manual: cualquier problema distinto de "good".
        suspicious_truth = reviewed_cases["manual_label"].ne("good")

        # Predicción equivalente a report["is_suspicious"].
        suspicious_prediction = (
            reviewed_cases["line_flag"]
            | reviewed_cases["bright_flag"]
        )

        suspicious_case_groups = {
            "Sospechosas: verdaderos positivos": reviewed_cases[
                suspicious_truth & suspicious_prediction
            ],
            "Sospechosas: falsos positivos": reviewed_cases[
                ~suspicious_truth & suspicious_prediction
            ],
            "Sospechosas: falsos negativos": reviewed_cases[
                suspicious_truth & ~suspicious_prediction
            ],
            "Sospechosas: verdaderos negativos": reviewed_cases[
                ~suspicious_truth & ~suspicious_prediction
            ],
        }

        true_positive = len(suspicious_case_groups["Sospechosas: verdaderos positivos"])
        false_positive = len(suspicious_case_groups["Sospechosas: falsos positivos"])
        false_negative = len(suspicious_case_groups["Sospechosas: falsos negativos"])
        true_negative = len(suspicious_case_groups["Sospechosas: verdaderos negativos"])

        precision = true_positive / max(true_positive + false_positive, 1)
        recall = true_positive / max(true_positive + false_negative, 1)
        accuracy = (true_positive + true_negative) / len(reviewed_cases)

        display(pd.DataFrame([{
            "casos_revisados": len(reviewed_cases),
            "TP": true_positive,
            "FP": false_positive,
            "FN": false_negative,
            "TN": true_negative,
            "precision": precision,
            "recall": recall,
            "accuracy": accuracy,
        }]).round(3))

        for title, cases in suspicious_case_groups.items():
            show_cases(cases, title)
    else:
        print("No hay etiquetas manuales revisadas para evaluar is_suspicious.")
else:
    print("No hay imágenes reales para evaluar todavía.")

## 20. Relación exploratoria con `PETRO_TH90` y conclusiones

La calidad visual no demuestra que `PETRO_TH90` sea erróneo. Esta última comparación solo busca asociaciones: si los scores se concentran en tamaños extremos, merece inspección adicional; no implica causalidad ni cambia la regla de FOV.

In [ ]:
if not quality_df.empty:
    catalog_columns = ["name", "PETRO_TH90", "Ttype", "label"]
    quality_catalog = quality_df.merge(sample_selected[catalog_columns], on="name", how="left")
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, score in zip(axes, ["line_score", "bright_score", "saturation_score"]):
        ax.scatter(quality_catalog["PETRO_TH90"], quality_catalog[score], c=quality_catalog["line_flag"], cmap="coolwarm", s=60)
        ax.set_xlabel("PETRO_TH90 (arcsec)")
        ax.set_ylabel(score)
        ax.set_title(f"{score} vs PETRO_TH90")
    plt.tight_layout()
    plt.show()

    quality_catalog.to_csv(QUALITY_METRICS_CSV, index=False)
    print(f"Métricas guardadas en: {QUALITY_METRICS_CSV}")
    display(quality_catalog[["name", "manual_label", "PETRO_TH90", "line_score", "bright_score", "saturation_score", "reasons"]])
else:
    print("Sin imágenes para relacionar con el catálogo.")

## 21. Prueba ampliada: 1,000 galaxias

Muestra estratificada de 500 galaxias tempranas y 500 tardías. Usa los mismos parámetros de descarga y calidad, pero guarda los resultados en un experimento separado. No hay etiquetas manuales a esta escala: el objetivo es obtener métricas, tasas de banderas y casos candidatos para inspección.

In [ ]:
LARGE_EXPERIMENT = "rgb_validation_1000"
LARGE_N_PER_CLASS = 500
LARGE_RANDOM_SEED = 2026

large_parts = []
for class_index, morphology_label in enumerate(["temprana", "tardia"]):
    candidates = pool[pool["label"].eq(morphology_label)]

    if len(candidates) < LARGE_N_PER_CLASS:
        raise ValueError(
            f"No hay suficientes galaxias de clase {morphology_label!r}: "
            f"{len(candidates)} disponibles, se requieren {LARGE_N_PER_CLASS}."
        )

    large_parts.append(
        candidates.sample(
            n=LARGE_N_PER_CLASS,
            random_state=LARGE_RANDOM_SEED + class_index,
        )
    )

large_sample = (
    pd.concat(large_parts, ignore_index=True)
    .sample(frac=1, random_state=LARGE_RANDOM_SEED)
    .reset_index(drop=True)
)

LARGE_OUTPUT_DIR = PLAYGROUND_DIR / LARGE_EXPERIMENT / "dr11_grz_fov6"
LARGE_INDEX_CSV = PLAYGROUND_DIR / LARGE_EXPERIMENT / "playground_index.csv"
LARGE_METRICS_CSV = PLAYGROUND_DIR / LARGE_EXPERIMENT / "quality_metrics.csv"
LARGE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Galaxias seleccionadas: {len(large_sample)}")
display(large_sample["label"].value_counts().rename_axis("label").to_frame("n"))

In [ ]:
large_jobs = prepare_jobs(
    sample_df=large_sample,
    playground_dir=PLAYGROUND_DIR,
    experiment=LARGE_EXPERIMENT,
    subfolder="dr11_grz_fov6",
    layer=LAYER,
    bands=BANDS,
    fov_factor=FOV_FACTOR,
    size_px=SIZE_PX,
    pixscale_min=PIXSCALE_MIN,
    pixscale_max=PIXSCALE_MAX,
)

large_index = download_batch(
    large_jobs,
    max_concurrent=MAX_CONCURRENT_REQUESTS,
    min_request_interval_s=MIN_REQUEST_INTERVAL_S,
    overwrite=False,
    timeout_s=TIMEOUT_S,
    max_retries=MAX_RETRIES,
    backoff_s=2.0,
    backoff_429_s=10.0,
    verbose=True,
)

large_index["file_exists"] = large_index["local_path"].map(
    lambda path: Path(path).is_file()
)
large_index.to_csv(LARGE_INDEX_CSV, index=False)

display(
    large_index["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .to_frame("n")
)

In [ ]:
large_samples = large_index.loc[
    large_index["file_exists"],
    ["name", "local_path"],
].rename(columns={"local_path": "path"}).copy()

large_samples["filename"] = large_samples["path"].map(lambda path: Path(path).name)
large_samples["manual_label"] = "unknown"

large_quality_df = evaluate_quality_table(large_samples)

large_quality_df["is_suspicious"] = (
    large_quality_df["line_flag"]
    | large_quality_df["bright_flag"]
)

large_quality_df = large_quality_df.merge(
    large_sample[["name", "label", "Ttype", "PETRO_TH90"]],
    on="name",
    how="left",
)

large_quality_df.to_csv(LARGE_METRICS_CSV, index=False)

print(f"Imágenes analizadas: {len(large_quality_df)}")
print(f"Métricas guardadas en: {LARGE_METRICS_CSV}")
display(
    large_quality_df[
        ["is_suspicious", "line_flag", "bright_flag", "saturation_flag"]
    ]
    .mean()
    .mul(100)
    .round(2)
    .rename("porcentaje")
    .to_frame()
)

In [ ]:
large_candidates = (
    large_quality_df[large_quality_df["is_suspicious"]]
    .sort_values(
        ["bright_score", "line_score", "saturation_score"],
        ascending=False,
    )
)

display(
    large_candidates[
        [
            "name",
            "label",
            "PETRO_TH90",
            "line_flag",
            "bright_flag",
            "saturation_flag",
            "line_score",
            "bright_score",
            "saturation_score",
            "reasons",
        ]
    ].head(5)
)

large_paths = {
    row.filename: row.path
    for _, row in large_samples.iterrows()
}

def show_large_cases(case_df, title, ncols=4):
    if case_df.empty:
        print(f"{title}: no hay casos.")
        return

    nrows = int(np.ceil(len(case_df) / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4 * ncols, 4 * nrows),
        squeeze=False,
    )

    for ax, (_, case) in zip(axes.flat, case_df.iterrows()):
        ax.imshow(load_rgb_image(large_paths[case["filename"]]))
        ax.set_title(
            f"{case['name']}\n"
            f"L={case['line_flag']} B={case['bright_flag']} S={case['saturation_flag']}",
            fontsize=8,
        )
        ax.axis("off")

    for ax in axes.flat[len(case_df):]:
        ax.axis("off")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_large_cases(
    large_candidates.head(16),
    "Primeros 16 candidatos sospechosos de la muestra de 1,000",
)

In [ ]:
# Clasificación automática en cuatro cuadrantes:
# filas = line_flag; columnas = bright_flag.
if large_quality_df.empty:
    print("No hay imágenes analizadas.")
else:
    MAX_PER_QUADRANT = 9
    QUADRANT_SEED = 2026

    quadrant_definitions = [
        (
            "Sin línea | Sin contaminación brillante",
            ~large_quality_df["line_flag"] & ~large_quality_df["bright_flag"],
        ),
        (
            "Sin línea | Con contaminación brillante",
            ~large_quality_df["line_flag"] & large_quality_df["bright_flag"],
        ),
        (
            "Con línea | Sin contaminación brillante",
            large_quality_df["line_flag"] & ~large_quality_df["bright_flag"],
        ),
        (
            "Con línea | Con contaminación brillante",
            large_quality_df["line_flag"] & large_quality_df["bright_flag"],
        ),
    ]

    fig, axes = plt.subplots(6, 6, figsize=(13, 13))
    axes = axes.reshape(2, 3, 2, 3)

    for quadrant_index, (title, mask) in enumerate(quadrant_definitions):
        cases = large_quality_df.loc[mask]

        if len(cases) > MAX_PER_QUADRANT:
            cases = cases.sample(
                n=MAX_PER_QUADRANT,
                random_state=QUADRANT_SEED + quadrant_index,
            )

        quadrant_row, quadrant_col = divmod(quadrant_index, 2)
        quadrant_axes = axes[quadrant_row, :, quadrant_col, :].flat

        for ax, (_, case) in zip(quadrant_axes, cases.iterrows()):
            ax.imshow(load_rgb_image(large_paths[case["filename"]]))
            ax.set_title(
                f"{case['name']}\n"
                f"L={int(case['line_flag'])} "
                f"B={int(case['bright_flag'])}",
                fontsize=7,
            )
            ax.axis("off")

        for ax in list(quadrant_axes)[len(cases):]:
            ax.axis("off")

        #quadrant_axes_list = list(quadrant_axes)
        #quadrant_axes_list[0].set_title(
        #    f"{title}\n"
        #    f"{mask.sum()} casos; se muestran {len(cases)}",
        #    fontsize=9,
        #    loc="left",
        #)

    fig.suptitle(
        "Muestra visual de clasificación automática: línea vs contaminación brillante",
        fontsize=13,
    )
    plt.tight_layout()
    plt.show()

## 22. Conclusiones

- Se desarrolló un flujo para descargar, validar e inspeccionar cutouts RGB de galaxias del Legacy Survey.

- El análisis usa imágenes JPEG renderizadas; las métricas describen propiedades visuales y no medidas fotométricas calibradas.

- **Detección de líneas (`line_flag`)**:
  - Identifica regiones brillantes, continuas, alargadas y extensas.
  - Usa perfiles por filas y columnas, continuidad de píxeles brillantes y componentes conectados.
  - En la muestra de 1,000 galaxias se activó en 1.9% de los casos.
  - Esto sugiere que no hay muchas galaxias con posibles artefactos lineales y que la regla es relativamente selectiva.

- **Contaminación brillante (`bright_flag`)**:
  - Identifica regiones brillantes fuera de la zona central del cutout.
  - Puede corresponder a estrellas, galaxias vecinas, halos o campos visualmente congestionados.
  - Se activó en 27.6% de la muestra ampliada.
  - Es la bandera más ambigua: algunas imágenes siguen siendo utilizables aunque contengan fuentes brillantes externas.

- Los campos con brillo externo no siempre son valores atípicos, pero pueden afectar una tarea de clasificación porque las fuentes cercanas pueden superponerse al objetivo o modificar su apariencia visual.

- Por esta razón, `bright_flag` debe conservarse como un indicador de calidad independiente. Según la tarea final, estos casos pueden excluirse, recibir menor peso o mantenerse con una etiqueta de contaminación.

- La bandera global `is_suspicious` se activó en 27.7% de la muestra.

- La muestra inicial pequeña permitió inspeccionar visualmente las reglas y ajustar parámetros; la muestra ampliada permitió comprobar su comportamiento sobre 1,000 galaxias.

- Para medir falsos positivos y falsos negativos con precisión se requeriría una submuestra revisada manualmente o un catálogo externo de referencia.